<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/main/notebooks/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data preprocessing

Tokenization + BIO labeling

## Relevant EDA Findings:

- Character-level annotations: loc_markers stores location spans as [start, end) offsets. Tokenization must preserve alignment with the original text.

- Annotation edge cases: EDA identified occasional whitespace at span boundaries, incorrect boundaries and annotation noise. These cases require attention during BIO label assignment.

- Multiword locations: 15.39% of UK and 11.86% of ru annotated spans contain multiple words. BIO labels must preserve entity boundaries across tokens.

- Mixed-language text: Ukrainian, russian, Latin/Cyrillic mixtures and homoglyphs occur in the data. Tokenization should handle them without modifying the original character positions.

- Long texts: Test texts are substantially longer than training texts and frequently contain line breaks, emojis, URLs and hashtags. Tokenization and sequence-length handling must account for these cases.

- Data leakage prevention: Tokenizer vocabulary must be trained on the training split only, using the existing validation_manifest_v2.parquet

## Environment Setup

In [16]:
import sys
import ast
import random
from pathlib import Path

import os
import base64
import subprocess

import numpy as np
import pandas as pd
from collections import Counter

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace, ByteLevel
from tokenizers.processors import ByteLevel as ByteLevelProcessor

In [2]:
if "google.colab" in sys.modules:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    root = "/content"
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    root = "/kaggle/working"

repo = "UA-Locations-NER-Deep-Learning"
branch = "main"
repo_dir = f"{root}/{repo}"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", repo_dir],
        env=env, check=True
    )

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Deep-Learning


In [3]:
sys.path.insert(0, repo_dir)

from src.data_io import get_data_dir, load_manifest, select_manifest_rows, load_data_splits
from src.preprocessing import preprocess_data
from src.eda_scan import parse_spans

splits_dir = Path(repo_dir) / "data" / "splits"

print("Modules imported successfully")
print("Validation manifest exists:", (splits_dir / "validation_manifest_v2.parquet").exists())

Modules imported successfully
Validation manifest exists: True


## Data Loading

In [4]:
ENVIRONMENT = "colab" if root == "/content" else "kaggle"

data_dir = get_data_dir(ENVIRONMENT)
print("Data directory:", data_dir)

100%|██████████| 811M/811M [00:21<00:00, 39.2MB/s]

Extracting files...


Data directory: /root/.cache/kagglehub/competitions/kse-ua-location-extraction-2026


Train/val/holdout splits loading:

In [5]:
splits = load_data_splits(ENVIRONMENT, splits_dir, manifest_name="validation_manifest_v2")

for name, df in splits.items():
    print(f"{name}: {len(df):,} texts")

uk_train: 805,983 texts
uk_val: 100,741 texts
uk_holdout: 100,757 texts
ru_train: 100,000 texts
ru_val: 24,626 texts
ru_holdout: 24,743 texts


## Tokenizer Configurations

Two ways to split text into tokens:
- **Whitespace BPE:** splits text into words and punctuation, then learns smaller word pieces.
- **ByteLevel BPE:** works with bytes and can represent unfamiliar characters, including emojis.

Both use a vocabulary of 32,000 tokens.

In [6]:
TOKENIZER_CONFIGS = {
    "bpe_whitespace": {
        "model": "BPE",
        "pre_tokenizer": "Whitespace",
        "vocab_size": 32_000,
        "special_tokens": ["[PAD]", "[UNK]"],
        "seed": 17
    },
    "bpe_bytelevel": {
        "model": "BPE",
        "pre_tokenizer": "ByteLevel",
        "vocab_size": 32_000,
        "special_tokens": ["[PAD]", "[UNK]"],
        "seed": 17
    }
}

for name, config in TOKENIZER_CONFIGS.items():
    print(name, "->", config)

bpe_whitespace -> {'model': 'BPE', 'pre_tokenizer': 'Whitespace', 'vocab_size': 32000, 'special_tokens': ['[PAD]', '[UNK]'], 'seed': 17}
bpe_bytelevel -> {'model': 'BPE', 'pre_tokenizer': 'ByteLevel', 'vocab_size': 32000, 'special_tokens': ['[PAD]', '[UNK]'], 'seed': 17}


In [7]:
def build_tokenizer(config):
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    pre_tokenizer = config["pre_tokenizer"]

    if pre_tokenizer == "Whitespace":
        tokenizer.pre_tokenizer = Whitespace()
        alphabet = []
    elif pre_tokenizer == "ByteLevel":
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        tokenizer.post_processor = ByteLevelProcessor(trim_offsets=True,add_prefix_space=False)
        alphabet = ByteLevel.alphabet()
    else:
        raise ValueError(f"Unknown pre-tokenizer: {pre_tokenizer}")

    trainer = BpeTrainer(vocab_size=config["vocab_size"], special_tokens=config["special_tokens"],
                         initial_alphabet=alphabet)
    return tokenizer, trainer

We feed texts from Ukrainian and Russian **train** datasets one by one for tokenizer training. Validation and holdout are not used.

In [9]:
def train_text_iterator(splits):
    for split_name in ("uk_train", "ru_train"):
        for text in splits[split_name]["text"]:
            yield str(text)

Checking that training texts are not empty

In [10]:
for name in ("uk_train", "ru_train"):
  empty = splits[name]["text"].fillna("").str.strip().eq("").sum()
  print(f"{name}: {empty} empty texts")

uk_train: 0 empty texts
ru_train: 0 empty texts


Training `bpe_whitespace`:

In [11]:
config = TOKENIZER_CONFIGS["bpe_whitespace"]
tokenizer_whitespace, trainer = build_tokenizer(config)

n_texts = len(splits["uk_train"]) + len(splits["ru_train"])
tokenizer_whitespace.train_from_iterator(train_text_iterator(splits),trainer=trainer,length=n_texts)

print("Tokenizer:", "bpe_whitespace")
print("Vocabulary size:", tokenizer_whitespace.get_vocab_size())
print("PAD ID:", tokenizer_whitespace.token_to_id("[PAD]"))
print("UNK ID:", tokenizer_whitespace.token_to_id("[UNK]"))

Tokenizer: bpe_whitespace
Vocabulary size: 32000
PAD ID: 0
UNK ID: 1


Checking its work

In [12]:
df = splits["uk_train"]
row = df.loc[df["loc_markers"] != "[]"].iloc[1]

text = row["text"]
spans = parse_spans(row["loc_markers"])
encoding_whitespace = tokenizer_whitespace.encode(text, add_special_tokens=False)

print("Text:", text)
print("Locations:", [text[start:end] for start, end in spans])
print("\nBPE tokens:")

for token, token_id, (start, end) in zip(encoding_whitespace.tokens, encoding_whitespace.ids, encoding_whitespace.offsets):
    print(f"{token:15} ID={token_id:5} [{start}, {end})")

Text: Нагадаємо, президент України Володимир Зеленський скликав позачергове засідання Верховної Ради на 17 березня.
Locations: ['України']

BPE tokens:
Нагадаємо       ID= 1204 [0, 9)
,               ID=   13 [9, 10)
президент       ID= 1284 [11, 20)
України         ID=  619 [21, 28)
Володимир       ID= 1945 [29, 38)
Зеленський      ID= 2212 [39, 49)
склика          ID= 7771 [50, 56)
в               ID=  252 [56, 57)
позачергове     ID=31974 [58, 69)
засідання       ID= 3013 [70, 79)
Верховної       ID= 3165 [80, 89)
Ради            ID= 2131 [90, 94)
на              ID=  414 [95, 97)
17              ID= 1459 [98, 100)
березня         ID= 2352 [101, 108)
.               ID=   15 [108, 109)


Training `bpe_bytelevel`

In [13]:
config = TOKENIZER_CONFIGS["bpe_bytelevel"]
tokenizer_bytelevel, trainer = build_tokenizer(config)

n_texts = len(splits["uk_train"]) + len(splits["ru_train"])
tokenizer_bytelevel.train_from_iterator(train_text_iterator(splits), trainer=trainer,length=n_texts)

print("Tokenizer: bpe_bytelevel")
print("Vocabulary size:", tokenizer_bytelevel.get_vocab_size())
print("PAD ID:", tokenizer_bytelevel.token_to_id("[PAD]"))
print("UNK ID:", tokenizer_bytelevel.token_to_id("[UNK]"))

Tokenizer: bpe_bytelevel
Vocabulary size: 32000
PAD ID: 0
UNK ID: 1


Checking its work

In [14]:
encoding_bytelevel = tokenizer_bytelevel.encode(text, add_special_tokens=False)

print("Text:", text)
print("Locations:", [text[start:end] for start, end in spans])
print("\nBPE tokens:")

for token, token_id, (start, end) in zip(encoding_bytelevel.tokens, encoding_bytelevel.ids, encoding_bytelevel.offsets):
    print(f"{token:15} ID={token_id:5} [{start}, {end})")

Text: Нагадаємо, президент України Володимир Зеленський скликав позачергове засідання Верховної Ради на 17 березня.
Locations: ['України']

BPE tokens:
ÐĿÐ°Ð³Ð°Ð´Ð°ÑĶÐ¼Ð¾ ID= 1326 [0, 9)
,               ID=   13 [9, 10)
ĠÐ¿ÑĢÐµÐ·Ð¸Ð´ÐµÐ½ÑĤ ID=  859 [11, 20)
ĠÐ£ÐºÑĢÐ°ÑĹÐ½Ð¸ ID=  604 [21, 28)
ĠÐĴÐ¾Ð»Ð¾Ð´Ð¸Ð¼Ð¸ÑĢ ID= 1574 [29, 38)
ĠÐĹÐµÐ»ÐµÐ½ÑģÑĮÐºÐ¸Ð¹ ID= 2827 [39, 49)
ĠÑģÐºÐ»Ð¸Ðº     ID= 8292 [50, 55)
Ð°Ð²            ID=  312 [55, 57)
ĠÐ¿Ð¾Ð·Ð°ÑĩÐµÑĢÐ³ ID=10070 [58, 66)
Ð¾Ð²Ðµ          ID= 2273 [66, 69)
ĠÐ·Ð°ÑģÑĸÐ´Ð°Ð½Ð½Ñı ID= 3283 [70, 79)
ĠÐĴÐµÑĢÑħÐ¾Ð²Ð½Ð¾ÑĹ ID= 3465 [80, 89)
ĠÐłÐ°Ð´Ð¸       ID= 2472 [90, 94)
ĠÐ½Ð°           ID=  331 [95, 97)
Ġ17             ID= 1882 [98, 100)
ĠÐ±ÐµÑĢÐµÐ·Ð½Ñı ID= 2555 [101, 108)
.               ID=   15 [108, 109)


## Tokenizer Evaluation

We compare both tokenizers on the same 5,000 training texts per language. We check token boundaries, sequence lengths and unknown tokens.

In [15]:
tokenizers = {
    "bpe_whitespace": tokenizer_whitespace,
    "bpe_bytelevel": tokenizer_bytelevel}
TOKENIZER_EVAL_SEED = 17
TOKENIZER_EVAL_SIZE = 5000

tokenizer_eval = {}
for name in ("uk_train", "ru_train"):
    tokenizer_eval[name] = splits[name].sample(n=min(TOKENIZER_EVAL_SIZE, len(splits[name])),random_state=TOKENIZER_EVAL_SEED)
    print(f"{name}: {len(tokenizer_eval[name]):,} texts")

uk_train: 5,000 texts
ru_train: 5,000 texts


Checking location boundaries:

Whether tokens preserve the start and end of each annotated location:
- exact — boundaries match
- partial — boundaries differ
- missing — no tokens cover the location

In [17]:
def check_span_alignment(encoding, spans): #per 1 text
  results = []

  for start, end in spans:
      overlapping = [(a, b) for a, b in encoding.offsets if a < end and b > start]
      if not overlapping:
          results.append("missing")
      elif overlapping[0][0] == start and overlapping[-1][1] == end:
          results.append("exact")
      else:
          results.append("partial")
  return results

In [18]:
def evaluate_alignment(tokenizer, df):
  counts = Counter()

  for row in df.itertuples(index=False):
      spans = parse_spans(row.loc_markers)
      if not spans:
          continue
      encoding = tokenizer.encode(row.text, add_special_tokens=False)
      counts.update(check_span_alignment(encoding, spans))

  return counts

In [19]:
rows = []
for split_name, df in tokenizer_eval.items():
  for name, tokenizer in tokenizers.items():
      counts = evaluate_alignment(tokenizer, df)
      rows.append({
          "dataset": split_name,
          "tokenizer": name,
          "exact": counts["exact"],
          "partial": counts["partial"],
          "missing": counts["missing"]
      })

pd.DataFrame(rows).set_index(["dataset", "tokenizer"])

exact  partial  missing
dataset  tokenizer                              
uk_train bpe_whitespace   1724        8        0
         bpe_bytelevel    1724        8        0
ru_train bpe_whitespace   2315        7        0
         bpe_bytelevel    2316        6        0

Inspecting partial alignments

In [24]:
for row_id, row in tokenizer_eval["uk_train"].iterrows():
  spans = parse_spans(row["loc_markers"])
  encoding = tokenizer_whitespace.encode(row["text"], add_special_tokens=False)
  if "partial" in check_span_alignment(encoding, spans):
      print("Example index:", row_id)
      break

Example index: 414678


In [26]:
row = splits["uk_train"].loc[414678]
example_text = row["text"]
example_spans = parse_spans(row["loc_markers"])

print("Text:", example_text)
print("Annotations:", [(example_text[a:b], (a, b)) for a, b in example_spans])

for name, tokenizer in tokenizers.items():
  print("\n", name)
  encoding = tokenizer.encode(example_text, add_special_tokens=False)
  for a, b in encoding.offsets:
    if any(a < end and b > start for start, end in example_spans):
        print(repr(example_text[a:b]), (a, b))

Text: Батальйон “Івано-Франківськ” отримав БТР від громади Заболотова – "Репортер"Війна, 30 січня.
Annotations: [('Івано-Франківсь', (11, 26)), ('Заболотова', (53, 63))]

 bpe_whitespace
'Івано' (11, 16)
'-' (16, 17)
'Франківськ' (17, 27)
'За' (53, 55)
'боло' (55, 59)
'това' (59, 63)

 bpe_bytelevel
'Івано' (11, 16)
'-' (16, 17)
'Франківськ' (17, 27)
'Заб' (53, 56)
'олот' (56, 60)
'ова' (60, 63)


The last letter is omitted in the annotation: «Івано-Франківсь» instead of «Івано-Франківськ».

Both tokenizers highlight the full word, so the boundaries do not match the annotation.

Here the problem is the annotation.

ru example:

In [28]:
df = splits["ru_train"]
matches = df[df["text"].str.contains("НаУкраину", regex=False, na=False)]
row = matches.iloc[0]
example_text = row["text"]
example_spans = parse_spans(row["loc_markers"])

print("Text:", example_text)
print("Annotations:", [(example_text[a:b], (a, b)) for a, b in example_spans])

for name, tokenizer in tokenizers.items():
  print("\n", name)
  encoding = tokenizer.encode(example_text, add_special_tokens=False)
  for a, b in encoding.offsets:
    if any(a < end and b > start for start, end in example_spans):
        print(repr(example_text[a:b]), (a, b))

Text: НаУкраину Виктор Янукович, поего словам, готов вернуться, когда «будут соблюдены условия его безопасности».
Annotations: [('Украину', (2, 9))]

 bpe_whitespace
'Украину' (2, 9)

 bpe_bytelevel
'аУ' (1, 3)
'У' (2, 3)
'кра' (3, 6)
'ину' (6, 9)


The text is missing a space: `НаУкраину`. Whitespace isolates `Украину` correctly, while a ByteLevel token also includes `а`, so the boundaries do not match the annotation.

Inspecting text length in tokens:

In [29]:
def count_sequence_lengths(tokenizer, df):
    return [len(tokenizer.encode(text, add_special_tokens=False).ids) for text in df["text"]]

for split_name, df in tokenizer_eval.items():
  print(f"\n{split_name}")
  for name, candidate_tokenizer in tokenizers.items():
      lengths = count_sequence_lengths(candidate_tokenizer, df)
      print(
          f"{name}:"
          f"mean={np.mean(lengths):.2f},"
          f"median={np.median(lengths):.1f},"
          f"p95={np.percentile(lengths, 95):.1f},"
          f"max={max(lengths)}")


uk_train
bpe_whitespace:mean=21.77,median=19.0,p95=42.0,max=70
bpe_bytelevel:mean=22.42,median=20.0,p95=43.0,max=75

ru_train
bpe_whitespace:mean=29.22,median=27.0,p95=55.0,max=136
bpe_bytelevel:mean=30.38,median=28.0,p95=57.0,max=147


Whitespace produces slightly shorter sequences than ByteLevel in both training samples

Checking [UNK]

In [30]:
def count_unknown_tokens(tokenizer, df):
  unk_id = tokenizer.token_to_id("[UNK]")
  total_tokens = 0
  unknown_tokens = 0

  for text in df["text"]:
    encoding = tokenizer.encode(text, add_special_tokens=False)
    total_tokens += len(encoding.ids)
    unknown_tokens += encoding.ids.count(unk_id)

  return unknown_tokens, total_tokens

In [31]:
for split_name, df in tokenizer_eval.items():
  print(f"\n{split_name}")

  for name, candidate_tokenizer in tokenizers.items():
    unknown, total = count_unknown_tokens(candidate_tokenizer, df)
    unk_pct = 100 * unknown / total if total else 0
    print(f"{name}: UNK={unknown}/{total} ({unk_pct:.4f}%)")


uk_train
bpe_whitespace: UNK=0/108864 (0.0000%)
bpe_bytelevel: UNK=0/112124 (0.0000%)

ru_train
bpe_whitespace: UNK=0/146085 (0.0000%)
bpe_bytelevel: UNK=0/151893 (0.0000%)


Impressive

Checking UNK tokens on test set:

In [32]:
test_df = pd.read_csv(data_dir / "test.csv")

for name, candidate_tokenizer in tokenizers.items():
  unknown, total = count_unknown_tokens(candidate_tokenizer, test_df)
  unk_pct = 100 * unknown / total if total else 0
  print(f"{name}: UNK={unknown}/{total} ({unk_pct:.4f}%)")

bpe_whitespace: UNK=861/42740 (2.0145%)
bpe_bytelevel: UNK=0/51013 (0.0000%)


**Tokenizer configuration conclusions:**

Whitespace produces slightly shorter sequences on the training samples, but 2.01% of its test tokens are [UNK]. ByteLevel has no unknown tokens on test. This gives it better character coverage, but does not guarantee better NER results.

## Converting spans to BIO labels

We assign a BIO label to each token: `B-LOC` for the start of a location, `I-LOC` for its continuation and `O` for other tokens.

Texts with mismatched tokens boundaries are excluded from training process, but are kept for validation and holdout evaluation.

In [33]:
def spans_to_bio(offsets, spans):
  labels = ["O"] * len(offsets)
  for start, end in spans:
    indices = [i for i, (a, b) in enumerate(offsets) if a < end and b > start]

    if not indices:
        raise ValueError(f"No tokens for span {(start, end)}")

    if offsets[indices[0]][0] != start or offsets[indices[-1]][1] != end:
        raise ValueError(f"Partial alignment for span {(start, end)}")

    labels[indices[0]] = "B-LOC"
    for i in indices[1:]:
        labels[i] = "I-LOC"

  return labels

Check

In [35]:
row = splits["uk_train"].loc[splits["uk_train"]["loc_markers"] != "[]"].iloc[1]
bio_text = row["text"]
bio_spans = parse_spans(row["loc_markers"])

encoding = tokenizer_whitespace.encode(bio_text, add_special_tokens=False)
labels = spans_to_bio(encoding.offsets, bio_spans)

print("Text:", bio_text)
for token, label in zip(encoding.tokens, labels):
  print(f"{token:20} {label}")

Text: Нагадаємо, президент України Володимир Зеленський скликав позачергове засідання Верховної Ради на 17 березня.
Нагадаємо            O
,                    O
президент            O
України              B-LOC
Володимир            O
Зеленський           O
склика               O
в                    O
позачергове          O
засідання            O
Верховної            O
Ради                 O
на                   O
17                   O
березня              O
.                    O


Multiword location check:

In [36]:
for row in splits["uk_train"].itertuples(index=False):
  spans = parse_spans(row.loc_markers)
  multiword_spans = [(start, end) for start, end in spans if len(row.text[start:end].split()) > 1]
  if multiword_spans:
      break

print("Text:", row.text)
print("Locations:", [row.text[start:end] for start, end in spans])

for name, candidate_tokenizer in tokenizers.items():
  encoding = candidate_tokenizer.encode(row.text, add_special_tokens=False)
  labels = spans_to_bio(encoding.offsets, spans)
  print(f"\n{name}")
  for token, label, (start, end) in zip(encoding.tokens, labels, encoding.offsets):
    if label != "O":
        print(f"{repr(row.text[start:end]):20} {label}")

Text: Подібні розіграші проводили в Великій Британії.
Locations: ['Великій Британії']

bpe_whitespace
'Великій'            B-LOC
'Британії'           I-LOC

bpe_bytelevel
'Великій'            B-LOC
'Британії'           I-LOC


## BIO Data Preparation

Function that prepares token ids, offsets and BIO labels for **each** text.

Texts with mismatched boundaries are skipped and their ids are saved in `skipped_ids`

This example shows how training records are prepared. Model notebooks use `prepare_ner_records` from `src/dataset.py`, which skips texts with partial alignments  only in training and keeps them for validation and holdout.

In [37]:
def prepare_bio_data(tokenizer, df, source):
  records = []
  skipped_ids = []

  for row_id, row in df.iterrows():
    spans = parse_spans(row["loc_markers"])
    encoding = tokenizer.encode(row["text"], add_special_tokens=False)
    try:
        labels = spans_to_bio(encoding.offsets, spans)
    except ValueError:
        skipped_ids.append(f"{source}_{row_id}")
        continue
    records.append({
        "text_id": f"{source}_{row_id}",
        "input_ids": encoding.ids,
        "offsets": encoding.offsets,
        "labels": labels
    })

  return records, skipped_ids

Full train BIO preparation, Whitespace BPE

In [38]:
uk_bio_whitespace, uk_skipped_whitespace = prepare_bio_data(tokenizer_whitespace,
                                                            splits["uk_train"],source="uk")
print("Prepared:", len(uk_bio_whitespace))
print("Skipped:", len(uk_skipped_whitespace))
print("Total:", len(uk_bio_whitespace) + len(uk_skipped_whitespace))

Prepared: 804367
Skipped: 1616
Total: 805983


In [39]:
ru_bio_whitespace, ru_skipped_whitespace = prepare_bio_data(tokenizer_whitespace, splits["ru_train"], source="ru")
print("Prepared:", len(ru_bio_whitespace))
print("Skipped:", len(ru_skipped_whitespace))
print("Total:", len(ru_bio_whitespace) + len(ru_skipped_whitespace))

Prepared: 99909
Skipped: 91
Total: 100000


## UNO reverse - BIO to spans

`bio_to_spans` turns BIO labels and token offsets into location spans, so we can extract names from the original text.

The function is stored in `src/preprocessing.py` and called by `predict` in `src/training.py` for validation, holdout and test predictions.

In [40]:
def bio_to_spans(labels, offsets):
  if len(labels) != len(offsets):
      raise ValueError("Labels and offsets must have equal lengths")

  spans = []
  start, end = None, None
  for label, (a, b) in zip(labels, offsets):
    if label == "B-LOC" or (label == "I-LOC" and start is None):
      if start is not None:
        spans.append((start, end))
      start, end = a, b

    elif label == "I-LOC":
      end = max(end, b)

    elif label == "O":
      if start is not None:
        spans.append((start, end))
        start, end = None, None
    else:
      raise ValueError(f"Unknown BIO label: {label}")

  if start is not None:
    spans.append((start, end))

  return spans

Full check (gold-spans -> BIO-labels -> restored spans)

In [41]:
results = []

for tokenizer_name, tokenizer in tokenizers.items():
  for split_name in ("uk_train", "ru_train"):
    df = splits[split_name]
    checked = skipped = mismatches = 0
    mismatch_ids = []

    for row_id, row in df.iterrows():
      spans = parse_spans(row["loc_markers"])
      encoding = tokenizer.encode(row["text"], add_special_tokens=False)

      try:
        labels = spans_to_bio(encoding.offsets, spans)
      except ValueError:
        skipped += 1
        continue

      restored = bio_to_spans(labels, encoding.offsets)
      checked += 1

      if sorted(spans) != sorted(restored):
        mismatches += 1
        if len(mismatch_ids) < 5:
            mismatch_ids.append(f"{split_name}_{row_id}")

    results.append({
        "tokenizer": tokenizer_name,
        "split": split_name,
        "checked": checked,
        "skipped": skipped,
        "mismatches": mismatches,
        "mismatch_ids": mismatch_ids})

roundtrip_results = pd.DataFrame(results)
display(roundtrip_results)

,tokenizer,split,checked,skipped,mismatches,mismatch_ids
0,bpe_whitespace,uk_train,804367,1616,0,[]
1,bpe_whitespace,ru_train,99909,91,0,[]
2,bpe_bytelevel,uk_train,804381,1602,0,[]
3,bpe_bytelevel,ru_train,99890,110,0,[]


Lost nothing during reverse conversion of BIO → spans (however, some of the texts were missed at the stage of spans → BIO)

On Validation

In [42]:
results_val = []

for tokenizer_name, tokenizer in tokenizers.items():
  for split_name in ("uk_val", "ru_val"):
    df = splits[split_name]
    checked = skipped = mismatches = 0
    mismatch_ids = []

    for row_id, row in df.iterrows():
      spans = parse_spans(row["loc_markers"])
      encoding = tokenizer.encode(row["text"], add_special_tokens=False)

      try:
        labels = spans_to_bio(encoding.offsets, spans)
      except ValueError:
        skipped += 1
        continue

      restored = bio_to_spans(labels, encoding.offsets)
      checked += 1

      if sorted(spans) != sorted(restored):
        mismatches += 1
        if len(mismatch_ids) < 5:
          mismatch_ids.append(f"{split_name}_{row_id}")

    results_val.append({
        "tokenizer": tokenizer_name,
        "split": split_name,
        "total": len(df),
        "checked": checked,
        "skipped": skipped,
        "mismatches": mismatches,
        "mismatch_ids": mismatch_ids})

roundtrip_val_results = pd.DataFrame(results_val)
display(roundtrip_val_results)

,tokenizer,split,total,checked,skipped,mismatches,mismatch_ids
0,bpe_whitespace,uk_val,100741,100572,169,0,[]
1,bpe_whitespace,ru_val,24626,24609,17,0,[]
2,bpe_bytelevel,uk_val,100741,100576,165,0,[]
3,bpe_bytelevel,ru_val,24626,24603,23,0,[]
